# starling — DFXM analysis walkthrough

Step-by-step interactive workflow for **strain sweep**, **mosaicity**, and **3D strain-mosa** scans.
Device is auto-detected: CUDA on ESRF GPU nodes → MPS on Apple Silicon → CPU fallback.

**How to use:** Edit the `CONFIG` cell (Section 1), then run top to bottom with *Run All*.

The fitting step is a single `dset.analyze(mask=grain)` call — dispatch is automatic by
motor-dimension count (1 motor → `Gauss1DResult`, ≥2 motors → `GaussNDResult`), so 1-D, 2-D and
3-D scans all "just work" and the 3-D scan produces **fitted** maps (centre / width / mosaicity),
not just moments. Results are named objects (`res.mu`, `res.fwhm`, `res.mosaicity()`,
`res.orientation()`) with a `.raw` array for back-compat.


## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import time, os, h5py

import starling
from starling import properties

FWHM = 2 * np.sqrt(2 * np.log(2))   # sigma → FWHM conversion factor

print(f'starling {starling.__version__}  |  device: {starling.get_device(None)}')


def list_scans(dataset_folder):
    """Print every scan command (scan_id ending in .1) found in the dataset folder."""
    folder = Path(dataset_folder)
    h5_files = sorted(folder.glob('*.h5'))
    if not h5_files:
        print(f'No .h5 file found in {folder}'); return
    h5 = h5_files[0]
    print(f'File: {h5.name}\n')
    with h5py.File(h5) as f:
        keys = sorted(
            [k for k in f.keys() if '.' in k and k[0].isdigit() and k.endswith('.1')],
            key=lambda s: int(s.split('.')[0])
        )
        print(f'{"scan_id":>8}  command')
        print('-' * 90)
        for sid in keys:
            try:
                cmd = f[sid]['title'][()].decode()
                # also show frame count
                def _find_data(name, obj):
                    if (isinstance(obj, h5py.Dataset)
                            and obj.ndim == 3 and obj.shape[1] > 100):
                        raise StopIteration(obj.shape[0])
                nf = '?'
                try: f[sid].visititems(_find_data)
                except StopIteration as e: nf = e.args[0]
                print(f'{sid:>8}  [{nf} frames]  {cmd}')
            except Exception:
                pass
    return h5


def make_symmetric_norm(arr, mask, n_sigma=3, fallback=40e-3):
    """Return (vmin, vmax) centred on median of arr[mask], ±n_sigma*std."""
    if mask.any():
        centre = np.nanmedian(arr[mask])
        spread = max(np.nanstd(arr[mask]), fallback)
        return centre - n_sigma * spread, centre + n_sigma * spread
    return np.nanmin(arr), np.nanmax(arr)

## 1 · Choose your scan  ← **EDIT THIS CELL**

Uncomment **one** option block (A, B, or C), set `RAW_DATA` for your system, then run.

In [ ]:
# ─── change this path for the ESRF cluster ────────────────────────────────────
RAW_DATA = '/Volumes/Elements/MA6278/RAW_DATA'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION A — Strain sweep  (fscan2d ccmth × mu)
# Each scan is one time-point in the charging series.  Edit scan_id to pick
# a different time-point ('2.1', '3.1', … '37.1').
# ═══════════════════════════════════════════════════════════════════════════════
DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_initial_strain_sweeps_during_charge_good'
SCAN_ID      = '1.1'
SCAN_TYPE    = 'strain_sweep'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION B — Mosaicity scan  (fscan2d chi × mu)
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_mosa_projection_charging_1'
# SCAN_ID      = '1.1'
# SCAN_TYPE    = 'mosa'

# ═══════════════════════════════════════════════════════════════════════════════
# OPTION C — 3D strain-mosa layer  (chi × mu × obpitch stack)
# Load all 11 obpitch steps as a single 5-D dataset.
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET_NAME = 'DFXM_insitu_repaired_cell_charging/DFXM_insitu_repaired_cell_charging_g1_2x_strain_mosa_layer_charging_2'
# SCAN_ID      = [f'{i}.1' for i in range(1, 12)]   # list → stacked load
# SCAN_TYPE    = 'strain_mosa_3d'

# ─── preprocessing settings (rarely need changing) ───────────────────────────
SIG_THRESHOLD = 50      # counts: pixels below this are masked in result maps
N_BG_FRAMES   = 5       # number of darkest frames averaged for background
HOT_SIGMA     = 5.0     # hot-pixel detection: σ above local 3×3 median
ROI_THRESHOLD = 0.05    # auto_roi: fraction of z-sum max defining the grain box
ROI_PAD       = 20      # pixels of padding around the grain box
CMAP_MDEG     = 40      # ±N mdeg symmetric range for orientation/strain maps

# ─── derived (do not edit) ────────────────────────────────────────────────────
_ds_base = Path(DATASET_NAME).name
H5_PATH  = f'{RAW_DATA}/{DATASET_NAME}/{_ds_base}.h5'
SCAN_MOTOR_3D = 'instrument/positioners/obpitch'

print(f'Dataset : {_ds_base}')
print(f'Scan ID : {SCAN_ID}')
print(f'Type    : {SCAN_TYPE}')
print(f'H5      : {H5_PATH}')
print(f'Exists  : {Path(H5_PATH).exists()}')

## 2 · Browse available scans *(optional)*

Run this cell to see every scan command in your dataset — useful for picking a different `SCAN_ID` in the CONFIG cell.

In [ ]:
list_scans(f'{RAW_DATA}/{DATASET_NAME}')

## 3 · Load scan

Data is read from the BLISS HDF5 master file into RAM.  
Typical times: ~25 s from USB3 drive; much faster from NVMe or network-attached ESRF storage.  
For the 3D option, a progress bar shows the 11-scan stack being assembled.

In [ ]:
t0 = time.perf_counter()

if SCAN_TYPE == 'strain_mosa_3d':
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID, scan_motor=SCAN_MOTOR_3D)
else:
    dset = starling.DataSet(H5_PATH, scan_id=SCAN_ID)

print(f'Loaded in {time.perf_counter()-t0:.1f} s')
print()
dset.info()
print()
print(f'data shape  : {dset.data.shape}')
print(f'dtype       : {dset.dtype}')
print(f'device      : {dset.device}')
sz_mb = dset.data.nbytes / 1e6
print(f'RAM in use  : {sz_mb:.0f} MB')

## 3b · Interactive denoise preview *(optional)*

Tune background / hot-pixel / ROI **live** and press **Apply** when happy. It is non-destructive — `dset.data` is untouched until Apply, and re-opening the widget always previews from the original data. If you use this, you can skip the manual preprocessing cells (4–6) below.


In [ ]:
%matplotlib widget
starling.viz.denoise_widget(dset, bg_n=N_BG_FRAMES, hot_sigma=HOT_SIGMA,
                            roi_threshold=ROI_THRESHOLD)
# (switch back with `%matplotlib inline` before the plotting cells below)


## 4 · Background subtraction

Background is estimated as the pixel-wise mean of the `N_BG_FRAMES` darkest frames — these are frames with the lowest total detector signal, which in a rocking scan are the off-Bragg frames where only readout noise is recorded.  
Subtraction is clamped so `uint16` never wraps.

In [ ]:
bg = dset.estimate_background(n_lowest=N_BG_FRAMES)
print(f'Background — mean: {bg.mean():.1f} cts   max: {int(bg.max())} cts   '
      f'using {N_BG_FRAMES} darkest frames')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(bg.astype(float), cmap='hot')
ax.set_title('Background image (mean of darkest frames)')
plt.colorbar(im, ax=ax, label='counts')
plt.tight_layout()
plt.show()

dset.subtract(bg)
print('Background subtracted.')

## 5 · ROI selection

The z-sum (detector integrated over all motor steps) reveals where the grain is.  
**Auto-ROI** thresholds at `ROI_THRESHOLD × max(z-sum)` and adds `ROI_PAD` pixels of padding.  

If the auto box misses the grain — uncomment the manual override block below and set your own pixel range.

In [ ]:
# Compute z-sum before cropping so we can preview the full detector
zsum_full = dset.data.reshape(*dset.data.shape[:2], -1).sum(-1).astype(float)
ny_full, nx_full = zsum_full.shape

# Propose the auto-ROI bounding box
roi = starling.preprocess.auto_roi(
    dset.data, threshold_rel=ROI_THRESHOLD, pad=ROI_PAD
)
r1, r2, c1, c2 = roi
pixel_mm = 6.5e-3   # mm per pixel (6.5 µm PCO pixel)
print(f'Auto ROI : rows {r1}–{r2}, cols {c1}–{c2}')
print(f'           {r2-r1} × {c2-c1} px = '
      f'{(r2-r1)*pixel_mm:.2f} × {(c2-c1)*pixel_mm:.2f} mm')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.imshow(zsum_full, cmap='hot', interpolation='nearest')
rect = plt.Rectangle((c1, r1), c2-c1, r2-r1,
                       edgecolor='cyan', facecolor='none', lw=2)
ax.add_patch(rect)
ax.set_title('Full detector z-sum  (cyan = auto ROI)')
ax.set_xlabel('detector column'); ax.set_ylabel('detector row')

ax = axes[1]
ax.imshow(zsum_full[r1:r2, c1:c2], cmap='hot', interpolation='nearest')
ax.set_title(f'Inside auto ROI  ({r2-r1}×{c2-c1} px)')
ax.set_xlabel('detector column'); ax.set_ylabel('detector row')

plt.tight_layout()
plt.show()

# ── Manual override — uncomment and edit if the auto box misses the grain ─────
# roi = (800, 1300, 900, 1500)   # (row_min, row_max, col_min, col_max)
# r1, r2, c1, c2 = roi
# print(f'Using manual ROI: {roi}')

# Apply ROI: crops dset.data in place
r1, r2, c1, c2 = roi
dset.data = np.ascontiguousarray(dset.data[r1:r2, c1:c2])
print(f'Data shape after crop: {dset.data.shape}')

## 6 · Hot-pixel removal *(optional — slow on large frames)*

Replaces isolated hot pixels with their 3×3 spatial median.  Detection threshold: `HOT_SIGMA × MAD` per frame.  
Skip this cell if the dataset is large and you do not see obvious streaks in the grain maps.

In [ ]:
n_frames = int(np.prod(dset.data.shape[2:]))
ny, nx = dset.data.shape[:2]
print(f'Running hot-pixel filter on {n_frames} frames of {ny}×{nx} px '
      f'— expect ~{n_frames*0.15:.0f} s on MPS/CPU ...')
t0 = time.perf_counter()
dset.remove_hot_pixels(n_sigma=HOT_SIGMA)
print(f'Done in {time.perf_counter()-t0:.1f} s')

## 7 · Grain overview

Z-sum after all preprocessing.  The signal mask (pixels above `SIG_THRESHOLD` in any frame) is used throughout to exclude background in result maps and statistics.

In [ ]:
ny, nx = dset.data.shape[:2]
zsum = dset.data.reshape(ny, nx, -1).sum(-1).astype(float)

# Signal mask: pixels where at least one frame exceeds the threshold
sig_mask = dset.data.reshape(ny, nx, -1).max(-1) > SIG_THRESHOLD

extent = [0, nx * pixel_mm, ny * pixel_mm, 0]   # mm

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(zsum, cmap='hot', extent=extent, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title('Grain — z-sum (integrated counts)')
plt.colorbar(im, ax=ax, label='counts')

ax = axes[1]
ax.imshow(sig_mask.astype(float), cmap='Greys_r', extent=extent,
           vmin=0, vmax=1, interpolation='nearest')
ax.set_xlabel('mm'); ax.set_ylabel('mm')
ax.set_title(f'Signal mask  (>{SIG_THRESHOLD} cts in any frame)')

plt.tight_layout()
plt.show()

print(f'Grain pixels : {sig_mask.sum():,} / {ny*nx:,}  '
      f'({100*sig_mask.mean():.1f} % of ROI)')

## 8 · Fitting

The correct fit is selected automatically from `SCAN_TYPE`:

| Type | Method | Output |
|------|--------|--------|
| `strain_sweep` | `fit_1D_gaussian` per µ layer | (ny, nx, 6, n_µ) — [A, σ, ccmth_centre, k, m, ok] |
| `mosa` | `moments` + `fit_2D_gaussian` | moments (ny,nx,2/2×2); gauss (ny,nx,8) |
| `strain_mosa_3d` | `moments` (3D) | (ny, nx, 3) + (ny, nx, 3, 3) |

> **Note:** the first `fit_1D_gaussian` / `fit_2D_gaussian` call per session pays a ~5–10 s `torch.compile` warm-up.  Subsequent calls use the cached kernel and are fast.

In [ ]:
# ── Grain mask: restrict fitting to grain pixels (faster; identical on-grain values) ──
grain = starling.preprocess.grain_mask(dset.data, threshold_rel=ROI_THRESHOLD)
print(f'grain mask: {grain.sum():,} / {grain.size:,} px ({100*grain.mean():.1f}%)')

from starling.properties import Gauss1DResult

if SCAN_TYPE == 'strain_sweep':
    # Depth-resolved strain: each µ layer is an independent 1-D ccmth rocking
    # scan, so fit per layer. (A single rocking scan would just be
    # `dset.analyze()` -> Gauss1DResult directly.)
    n_ccmth, n_mu = dset.data.shape[2], dset.data.shape[3]
    mu_values = dset.motors[1, 0, :]
    print(f'Strain sweep: {n_ccmth} ccmth × {n_mu} µ layers '
          f'(compile warm-up on first layer ~5-10 s)...')
    res = []
    for k in range(n_mu):
        t0 = time.perf_counter()
        p = properties.fit_1D_gaussian(dset.data[:, :, :, k], dset.motors[:1, :, k],
                                       mask=grain, device=dset.device)
        r = Gauss1DResult.from_raw(p)
        ok_k = (r.success > 0.5) & grain & (r.A > SIG_THRESHOLD)
        print(f'  µ layer {k:2d}  µ={mu_values[k]:.3f}°  success: {ok_k.sum():5,}  '
              f'({time.perf_counter()-t0:.2f} s)')
        res.append(r)

else:
    # Mosa (2 motors) and 3D strain-mosa (3 motors) dispatch automatically:
    # analyze() returns a GaussNDResult — the 3D scan produces FITTED maps,
    # not just moments.
    print(f'analyze(): {dset.n_motor_dims} motor dims '
          f'(compile warm-up on first call ~5-10 s)...')
    t0 = time.perf_counter()
    res = dset.analyze(mask=grain)
    print(f'  -> {type(res).__name__}  D={getattr(res, "D", "-")}  '
          f'success {100*(res.success > 0).mean():.1f}%  ({time.perf_counter()-t0:.2f} s)')


## 9 · Result maps

All maps are masked to `sig_mask` (grain pixels only); background shows as white.  
Orientation / strain maps are displayed as **deviations from the median** in **millidegrees**.

In [ ]:
# helper: nan-mask outside a boolean mask
def masked(arr, m):
    out = arr.astype(float).copy()
    out[~m] = np.nan
    return out

def dev_mdeg(arr, m):
    return (arr - np.nanmedian(arr[m])) * 1000


# ─── STRAIN SWEEP — per-µ-layer 1D fits (Gauss1DResult list) ─────────────────
if SCAN_TYPE == 'strain_sweep':
    n_mu = len(res)
    n_show = min(n_mu, 6)
    k_show = np.round(np.linspace(0, n_mu - 1, n_show)).astype(int)
    fig, axes = plt.subplots(3, n_show, figsize=(3.5 * n_show, 9), constrained_layout=True)
    if n_show == 1:
        axes = axes[:, None]
    for col, k in enumerate(k_show):
        r = res[k]
        ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
        im = axes[0, col].imshow(masked(dev_mdeg(r.mu, ok), ok), cmap='RdBu_r',
                                 vmin=-CMAP_MDEG, vmax=CMAP_MDEG, interpolation='nearest')
        axes[0, col].set_title(f'µ={mu_values[k]:.3f}°', fontsize=9)
        plt.colorbar(im, ax=axes[0, col], label='mdeg' if col == n_show - 1 else '')
        if col == 0:
            axes[0, col].set_ylabel('Δccmth (mdeg)')
        im = axes[1, col].imshow(masked(r.fwhm, ok), cmap='viridis', vmin=0, vmax=0.02,
                                 interpolation='nearest')
        plt.colorbar(im, ax=axes[1, col], label='°' if col == n_show - 1 else '')
        if col == 0:
            axes[1, col].set_ylabel('FWHM (°)')
        im = axes[2, col].imshow(masked(r.A, ok), cmap='hot', vmin=0, interpolation='nearest')
        plt.colorbar(im, ax=axes[2, col], label='cts' if col == n_show - 1 else '')
        if col == 0:
            axes[2, col].set_ylabel('Amplitude (cts)')
    for row in axes:
        for ax in row:
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'Strain sweep maps — {_ds_base}', y=1.01)
    plt.show()

    med_centre, med_fwhm = [], []
    for r in res:
        ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
        med_centre.append(np.nanmedian(r.mu[ok]) if ok.any() else np.nan)
        med_fwhm.append(np.nanmedian(r.fwhm[ok]) if ok.any() else np.nan)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(mu_values, med_centre, 'o-', color='steelblue', lw=2, ms=6)
    axes[0].set_xlabel('µ (°)'); axes[0].set_ylabel('Median ccmth peak centre (°)')
    axes[0].set_title('Strain depth profile'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(mu_values, med_fwhm, 's-', color='coral', lw=2, ms=6)
    axes[1].set_xlabel('µ (°)'); axes[1].set_ylabel('Median FWHM (°)')
    axes[1].set_title('Rocking width vs depth'); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


# ─── MOSA — 2D Gaussian (GaussNDResult, D=2) ─────────────────────────────────
elif SCAN_TYPE == 'mosa':
    ok = (res.success > 0.5) & sig_mask
    # centre = ORIENTATION (first moment); FWHM/spread = MOSAICITY (second moment)
    maps_mosa = [
        (masked(dev_mdeg(res.mu[..., 0], ok), ok), 'RdBu_r', -CMAP_MDEG, CMAP_MDEG, 'Δchi orientation (mdeg)'),
        (masked(dev_mdeg(res.mu[..., 1], ok), ok), 'RdBu_r', -CMAP_MDEG, CMAP_MDEG, 'Δµ orientation (mdeg)'),
        (masked(res.A, ok), 'hot', 0, None, 'Amplitude (cts)'),
        (masked(res.fwhm[..., 0] * 1000, ok), 'viridis', 0, None, 'FWHM chi (mdeg)'),
        (masked(res.fwhm[..., 1] * 1000, ok), 'viridis', 0, None, 'FWHM µ (mdeg)'),
        (masked(res.mosaicity(mode='scalar') * FWHM * 1000, ok), 'plasma', 0, None,
         'Mosaicity = orientation spread (mdeg FWHM)'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    for ax, (data, cmap, vmin, vmax, title) in zip(axes.flat, maps_mosa):
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.85)
    fig.suptitle(f'Mosaicity maps — {_ds_base}  (centre = orientation, spread = mosaicity)')
    plt.show()

    # classic orientation colour image — explicitly the MEAN orientation, not mosaicity
    sel, rgb, key = res.orientation(axes=(0, 1), as_rgb=True)
    rgb_disp = np.where(sig_mask[..., None], rgb, 1.0)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'width_ratios': [3, 1]})
    axs[0].imshow(rgb_disp); axs[0].set_title('Orientation map (hue=direction, sat=|deviation|)')
    axs[0].set_xticks([]); axs[0].set_yticks([])
    axs[1].imshow(key); axs[1].set_title('colour key'); axs[1].set_xticks([]); axs[1].set_yticks([])
    plt.tight_layout(); plt.show()


# ─── 3D STRAIN-MOSA — FITTED maps (GaussNDResult, D=3) ───────────────────────
elif SCAN_TYPE == 'strain_mosa_3d':
    ob_values = dset.motors[2, 0, 0, :]
    n_ob = dset.data.shape[4]
    ok = (res.success > 0.5) & sig_mask

    # intensity depth profile (where the grain sits in obpitch)
    I_per_layer = dset.data.sum(axis=(0, 1, 2, 3))
    peak_layer = int(np.argmax(I_per_layer))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ob_values, I_per_layer / I_per_layer.max(), 'o-', color='steelblue', lw=2, ms=7)
    ax.axvline(ob_values[peak_layer], ls='--', color='crimson', lw=1.5,
               label=f'peak: {ob_values[peak_layer]:.4f}° (layer {peak_layer+1})')
    ax.set_xlabel('obpitch (°)'); ax.set_ylabel('Normalised integrated intensity')
    ax.set_title('Depth profile — intensity vs obpitch layer'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    # fitted centre maps (chi, µ, obpitch) — from the 3D Gaussian fit
    labels = ['chi centre', 'µ centre', 'obpitch centre']
    cmaps = ['RdBu_r', 'RdBu_r', 'plasma']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    for i in range(3):
        val = masked(res.mu[..., i], ok)
        if ok.any():
            centre = np.nanmedian(val[ok]); spread = max(np.nanstd(val[ok]), 1e-3)
            vmin = centre - 3 * spread if i < 2 else np.nanmin(val[ok])
            vmax = centre + 3 * spread if i < 2 else np.nanmax(val[ok])
        else:
            vmin = vmax = None
        im = axes[i].imshow(val, cmap=cmaps[i], vmin=vmin, vmax=vmax, interpolation='nearest')
        axes[i].set_title(labels[i]); axes[i].set_xticks([]); axes[i].set_yticks([])
        plt.colorbar(im, ax=axes[i], label='°')
    fig.suptitle(f'3D fitted centre maps — {_ds_base}')
    plt.show()

    # mosaicity from the chi,µ orientation block only (excludes obpitch/strain axis)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    im = axes[0].imshow(masked(res.mosaicity(axes=(0, 1), mode='scalar') * FWHM * 1000, ok),
                        cmap='plasma', interpolation='nearest')
    axes[0].set_title('Mosaicity (chi,µ spread, mdeg FWHM)')
    axes[0].set_xticks([]); axes[0].set_yticks([]); plt.colorbar(im, ax=axes[0])
    im = axes[1].imshow(masked(res.A, ok), cmap='hot', interpolation='nearest')
    axes[1].set_title('Amplitude (cts)')
    axes[1].set_xticks([]); axes[1].set_yticks([]); plt.colorbar(im, ax=axes[1])
    fig.suptitle(f'3D mosaicity + amplitude — {_ds_base}')
    plt.show()


## 10 · Statistics

Distributions over all grain pixels (masked to `sig_mask`).  Each histogram shows the **spread within a single grain image** — the width reflects real orientation / strain heterogeneity inside the grain.

In [ ]:
print('\n' + '─' * 60)

if SCAN_TYPE == 'strain_sweep':
    best_k = max(range(len(res)),
                 key=lambda k: ((res[k].success > 0.5) & sig_mask
                                & (res[k].A > SIG_THRESHOLD)).sum())
    r = res[best_k]
    ok = (r.success > 0.5) & sig_mask & (r.A > SIG_THRESHOLD)
    centres_deg, fwhm_deg, amps = r.mu[ok], r.fwhm[ok], r.A[ok]
    print(f'Strain sweep — best µ layer: k={best_k}  µ={mu_values[best_k]:.3f}°')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  ccmth peak centre          : {np.median(centres_deg):.5f}°  '
          f'±{np.std(centres_deg)*1000:.2f} mdeg spread')
    print(f'  Median FWHM                : {np.median(fwhm_deg)*1000:.2f} mdeg')
    print(f'  Median amplitude           : {np.median(amps):.0f} counts')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist((centres_deg - np.median(centres_deg)) * 1000, bins=80,
                 color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δccmth from median (mdeg)'); axes[0].set_ylabel('Grain pixels')
    axes[0].set_title(f'Strain distribution — µ={mu_values[best_k]:.3f}°'); axes[0].grid(True, alpha=0.2)
    axes[1].hist(fwhm_deg * 1000, bins=80, color='coral', edgecolor='none', alpha=0.85)
    axes[1].set_xlabel('FWHM (mdeg)'); axes[1].set_ylabel('Grain pixels')
    axes[1].set_title('Rocking width distribution'); axes[1].grid(True, alpha=0.2)
    axes[2].plot(mu_values,
                 [(res[k].A[sig_mask].mean() if sig_mask.any() else 0) for k in range(len(res))],
                 'o-', color='seagreen', lw=2)
    axes[2].set_xlabel('µ (°)'); axes[2].set_ylabel('Mean amplitude (cts)')
    axes[2].set_title('Signal strength vs depth'); axes[2].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'mosa':
    ok = (res.success > 0.5) & sig_mask
    chi_px, mu_px = res.mu[..., 0][ok], res.mu[..., 1][ok]
    fchi_px, fmu_px = res.fwhm[..., 0][ok] * 1000, res.fwhm[..., 1][ok] * 1000
    mosa_px = res.mosaicity(mode='scalar')[ok] * FWHM * 1000
    print('Mosaicity statistics:')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    print(f'  chi orientation centre     : {np.median(chi_px):.4f}°  ±{np.std(chi_px)*1000:.2f} mdeg')
    print(f'  µ orientation centre       : {np.median(mu_px):.4f}°  ±{np.std(mu_px)*1000:.2f} mdeg')
    print(f'  Median FWHM chi            : {np.median(fchi_px):.1f} mdeg')
    print(f'  Median FWHM µ              : {np.median(fmu_px):.1f} mdeg')
    print(f'  Median total mosaicity     : {np.median(mosa_px):.1f} mdeg (orientation spread)')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist((chi_px - np.median(chi_px)) * 1000, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
    axes[0].axvline(0, color='k', lw=1.5, ls='--')
    axes[0].set_xlabel('Δchi orientation (mdeg)'); axes[0].set_ylabel('Grain pixels')
    axes[0].set_title('chi orientation spread'); axes[0].grid(True, alpha=0.2)
    axes[1].hist((mu_px - np.median(mu_px)) * 1000, bins=80, color='coral', edgecolor='none', alpha=0.85)
    axes[1].axvline(0, color='k', lw=1.5, ls='--')
    axes[1].set_xlabel('Δµ orientation (mdeg)'); axes[1].set_title('µ orientation spread'); axes[1].grid(True, alpha=0.2)
    axes[2].hist(fchi_px, bins=80, color='seagreen', edgecolor='none', alpha=0.7, label='chi')
    axes[2].hist(fmu_px, bins=80, color='purple', edgecolor='none', alpha=0.5, label='µ')
    axes[2].set_xlabel('FWHM (mdeg)'); axes[2].set_title('Rocking-curve width distributions')
    axes[2].legend(); axes[2].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


elif SCAN_TYPE == 'strain_mosa_3d':
    ok = (res.success > 0.5) & sig_mask
    print('3D strain-mosa statistics (from the fitted 3D Gaussian):')
    print(f'  Grain pixels with good fit : {ok.sum():,}')
    for i, lbl in enumerate(['chi', 'µ', 'obpitch']):
        v = res.mu[..., i][ok]
        print(f'  {lbl} centre : {np.median(v):.4f}°  ±{np.std(v)*1000:.2f} mdeg spread')
    mos = res.mosaicity(axes=(0, 1), mode='scalar')[ok] * FWHM * 1000
    print(f'  chi,µ mosaicity (spread)   : {np.median(mos):.1f} mdeg FWHM')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors_3d = ['steelblue', 'coral', 'seagreen']
    for i, lbl in enumerate(['chi centre', 'µ centre', 'obpitch centre']):
        v = res.mu[..., i][ok]
        axes[i].hist((v - np.median(v)) * 1000, bins=80, color=colors_3d[i], edgecolor='none', alpha=0.85)
        axes[i].axvline(0, color='k', lw=1.5, ls='--')
        axes[i].set_xlabel(f'Δ{lbl} (mdeg)'); axes[i].set_ylabel('Grain pixels')
        axes[i].set_title(f'{lbl} distribution'); axes[i].grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()


## 11 · Save results

In [ ]:
OUT = f'/Users/matt/Lab/projects/DFXM/analysis/notebook_{SCAN_TYPE}'
os.makedirs(OUT, exist_ok=True)

np.save(f'{OUT}/sig_mask.npy', sig_mask)
np.save(f'{OUT}/grain_mask.npy', grain)
np.save(f'{OUT}/motors.npy', dset.motors)

if SCAN_TYPE == 'strain_sweep':
    # stack per-µ-layer Gauss1DResult.raw -> (ny, nx, 6, n_mu)
    np.save(f'{OUT}/params.npy', np.stack([r.raw for r in res], axis=-1))
else:
    # named result object (A / mu / cov / c / success) -> one HDF5
    res.to_h5(f'{OUT}/result.h5')

print(f'Saved to {OUT}/')
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{f}') / 1e6
    print(f'  {f:<35}  {sz:.1f} MB')
